In [1]:
import pandas as pd

data_info_dir = "/mnt/l/Basic/divi/jstoker/slicer_pdac/Marius/Reza Morphology/data/preprocessed_data/CRLM/dataset_CAESAR_dec23_MASTER_incl morphologypathology.xlsx"
raw_data_df = pd.read_excel(data_info_dir)

filtered_by_segmentation = raw_data_df[raw_data_df['Segmented'] == 'Yes']

print(f"Total cases: {len(filtered_by_segmentation)}")

Total cases: 418


In [2]:
df = filtered_by_segmentation.copy()

mut_map = {
    "BRAF mutation": 0,
    "RAS & BRAF wildtype": 1,
    "RAS mutation": 2,
}
sex_map = {"Female": 0, "Male": 1}

pathology_map = {"nan": -1,
                "No histological response": 0,
                "Partial histological response": 1,
                "Major histological response": 1}  

morph_response_map = {"No response": 0, "Optimal response": 1, "Suboptimal response": 2, "Unknown": -1}
morphscore_map = {"Unknown": -1, 1:0, 2:1, 3:2}

# Map / coerce
df["mutstat_enc"] = df["mutstat"].map(mut_map).fillna(-1).astype(int)
df["sex_enc"] = df["sex"].map(sex_map).fillna(-1).astype(int)
df["who_enc"] = pd.to_numeric(df["WHO"], errors="coerce").fillna(-1).astype(int)
df["age_f"] = pd.to_numeric(df["Age"], errors="coerce").fillna(-1.0).astype(float)
df["baseline_ttv"] = pd.to_numeric(df["Baseline volume ml"], errors="coerce").fillna(-1.0).astype(float)
df["delta_ttv_rel"] = pd.to_numeric(df["FU1 delta vol rel"], errors="coerce").fillna(-1.0).astype(float)

df["pathology_result"] = df["Pathology"].fillna("nan").map(pathology_map).astype(int)
df["morph_response"] = df["morphresponse_best"].map(morph_response_map).fillna(-1).astype(int)
df["morph_score_base"] = df["morphscorebase_majority"].map(morphscore_map).fillna(-1).astype(int)
df["morph_score_followup"] = df["morphscorefirstfu_majority"].map(morphscore_map).fillna(-1).astype(int)
df["early_recurrence"] = pd.to_numeric(df["ER (1 = yes, 0 = no)"], errors="coerce").fillna(0).astype(int)

osm = pd.to_numeric(df["OSm"], errors="coerce")
df["overall_survival_24m"] = (osm > 24).fillna(False).astype(int)

In [5]:
import pandas as pd
import numpy as np
from itertools import combinations
from scipy.stats import chi2_contingency
from sklearn.metrics import mutual_info_score, normalized_mutual_info_score

list_of_labels = [
    "overall_survival_24m",
    "pathology_result",
    "early_recurrence",
    "morph_response",
    "morph_score_base",
    "morph_score_followup",
]

def cramers_v(x, y):
    tab = pd.crosstab(x, y)
    if tab.shape[0] < 2 or tab.shape[1] < 2:
        return np.nan, np.nan

    chi2, p, _, _ = chi2_contingency(tab)
    n = tab.to_numpy().sum()
    r, k = tab.shape
    v = np.sqrt(chi2 / (n * min(r - 1, k - 1)))
    return v, p

def label_correlation_table(df, label_cols, missing_value=-1):
    rows = []

    for a, b in combinations(label_cols, 2):
        sub = df[[a, b]].copy()
        sub = sub[(sub[a] != missing_value) & (sub[b] != missing_value)].dropna()

        n = len(sub)
        if n == 0:
            rows.append({
                "label_a": a,
                "label_b": b,
                "n": 0,
                "cramers_v": np.nan,
                "mutual_info": np.nan,
                "normalized_mutual_info": np.nan,
                "chi2_pvalue": np.nan,
            })
            continue

        x = sub[a]
        y = sub[b]

        v, p = cramers_v(x, y)
        mi = mutual_info_score(x, y)
        nmi = normalized_mutual_info_score(x, y)

        rows.append({
            "label_a": a,
            "label_b": b,
            "n": n,
            "cramers_v": v,
            "mutual_info": mi,
            "normalized_mutual_info": nmi,
            "chi2_pvalue": p,
        })

    out = pd.DataFrame(rows)
    out = out.sort_values(["cramers_v", "normalized_mutual_info"], ascending=False)
    return out.reset_index(drop=True)

corr_table = label_correlation_table(df, list_of_labels, missing_value=-1)
display(corr_table.round(4))

,label_a,label_b,n,cramers_v,mutual_info,normalized_mutual_info,chi2_pvalue
0,morph_response,morph_score_followup,403,0.6198,0.3659,0.3607,0.0000
1,morph_response,morph_score_base,410,0.3425,0.1636,0.1874,0.0000
2,morph_score_base,morph_score_followup,403,0.2700,0.0723,0.0762,0.0000
3,pathology_result,early_recurrence,201,0.2245,0.0279,0.0406,0.0015
4,pathology_result,morph_score_base,201,0.1418,0.0100,0.0134,0.1326
5,early_recurrence,morph_response,410,0.1107,0.0061,0.0078,0.0813
6,pathology_result,morph_score_followup,199,0.0838,0.0035,0.0040,0.4976
7,early_recurrence,morph_score_followup,411,0.0782,0.0030,0.0035,0.2844
8,overall_survival_24m,morph_response,410,0.0759,0.0029,0.0036,0.3072
9,overall_survival_24m,early_recurrence,418,0.0650,0.0025,0.0038,0.1839
